# Diffusion data augmentation for skin lesion classification (Colab)

Runtime → Change runtime type → **T4 GPU**. Expect several hours in total: about 30–60 minutes of LoRA training per rare class, image generation, then 4 classifier runs.

This is a learning project, not a diagnostic tool.

In [ ]:
!git clone https://github.com/FloraP277/Gen-AI---Agentic-Systems.git
%cd "Gen-AI---Agentic-Systems/Generative-AI-&-Agentic-Systems/skin-lesion-diffusion"
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, 'src')

## 1. Get HAM10000
Option A: upload your `kaggle.json` (Kaggle → Settings → API) and download the Kaggle copy. Option B: download from Harvard Dataverse (doi:10.7910/DVN/DBW86T) and unzip into `data/ham10000`.

In [ ]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/ham10000 --unzip

## 2. Lesion-level splits and diffusion training sets

In [ ]:
from sld import prepare
prepare.main(['--root', 'data/ham10000', '--out', 'work'])

## 3. Fine-tune Stable Diffusion with LoRA, one rare class at a time
Uses the official Hugging Face diffusers example script for the installed version.

In [ ]:
import diffusers, json
v = diffusers.__version__
!wget -q -O train_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{v}/examples/text_to_image/train_text_to_image_lora.py
rare = json.load(open('work/rare.json'))['rare_classes']
rare

In [ ]:
for dx in rare:
    !accelerate launch train_lora.py \
      --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 \
      --train_data_dir=work/diffusion_train/{dx} --caption_column=text \
      --resolution=512 --random_flip --center_crop \
      --train_batch_size=1 --gradient_accumulation_steps=4 --max_train_steps=1500 \
      --learning_rate=1e-04 --lr_scheduler=cosine --lr_warmup_steps=0 \
      --mixed_precision=fp16 --seed=42 --output_dir=work/lora/{dx}

## 4. Generate synthetic images
Enough for the largest ratio you will test (2 synthetic per real image).

In [ ]:
import pandas as pd
from sld import generate
splits = pd.read_csv('work/splits.csv')
for dx in rare:
    n = 2 * int(((splits.split == 'train') & (splits.dx == dx)).sum())
    generate.main(['--lora', f'work/lora/{dx}', '--dx', dx, '--n', str(n), '--out', 'work/synthetic'])

## 5. Real vs synthetic, side by side

In [ ]:
import glob, matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(len(rare), 8, figsize=(16, 2.2 * len(rare)))
for r, dx in enumerate(rare):
    real = splits[(splits.split == 'train') & (splits.dx == dx)].path.tolist()[:4]
    fake = sorted(glob.glob(f'work/synthetic/{dx}/*.jpg'))[:4]
    for c, (p, label) in enumerate([(x, 'real') for x in real] + [(x, 'synthetic') for x in fake]):
        axes[r, c].imshow(Image.open(p)); axes[r, c].set_title(f'{dx} {label}', fontsize=8); axes[r, c].axis('off')
plt.tight_layout(); plt.savefig('work/results_grid.png', dpi=120)

## 6. The experiment: classifier with 0, 0.5, 1 and 2 synthetic images per real rare-class image

In [ ]:
from sld import experiment
experiment.main(['--work', 'work', '--ratios', '0', '0.5', '1', '2', '--epochs', '8', '--seeds', '0', '1'])